In [2]:
import polars
%load_ext autoreload
%autoreload 2
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.axes as axes

import polars as pl
import adbc_driver_postgresql.dbapi as dbapi


def gen_heatmap(cards: int) -> axes.Axes:
    with dbapi.connect("host=dell1 port=5432 dbname=bingo user=tokeiya3") as conn:
        df = pl.read_database("""
                              WITH tbl AS (SELECT RANK() OVER (PARTITION BY play_id ORDER BY round) AS rank,
                                                  round,
                                                  hit_count
                                           FROM all_cards
                                           WHERE cards = $1)
                              SELECT rank, round, SUM(hit_count) AS hit_count
                              FROM tbl
                              GROUP BY rank, round
                              ORDER BY rank, round
                              """, connection=conn, execute_options={'parameters': [cards]})

    hdf = df.pivot(index='round', on='rank', values='hit_count', aggregate_function='sum').fill_null(0).sort('round')
    plt.figure(figsize=(20, 10), dpi=200)
    ax = sns.heatmap(hdf)
    ax.invert_yaxis()

    ax.set_xlabel('Rank')
    ax.set_ylabel('Round')
    ax.set_title(f'{cards} players')
    return ax

In [ ]:
from typing_extensions import cast
from matplotlib.figure import Figure

for i in [x * 5 for x in range(1, 21)]:
    cast(Figure, gen_heatmap(i).figure).savefig(f'../images/rank_round_{i}.png', format='png', bbox_inches=None)
    print(f'heatmap_{i}.png saved')

In [10]:
from PIL import Image

paths: list[str] = []

for i in [x * 5 for x in range(1, 21)]:
    paths.append(f'../images/rank_round_{i}.png')

images = [Image.open(path).convert('RGBA') for path in paths]

images[0].save(
    '../images/rank_round.gif',
    save_all=True,
    append_images=images[1:],
    duration=300,
    loop=0,
    lossless=True
)

In [11]:
del arr
del i
del images
del paths
